# Project 7 — Local Research Paper Explainer

## Explain Papers in Plain English

**Goal:** Take a research paper (sections, abstract, key claims) and produce
beginner-friendly explanations, glossary, and takeaway cards.

**Stack:** Ollama · LangChain · Jupyter

### What You'll Learn
1. Section-aware summarization
2. Difficulty-level prompting (beginner / practitioner / expert)
3. Glossary extraction from technical text
4. Key takeaway generation

### Prerequisites
- **Ollama** running with a chat model (e.g., `qwen3:8b`)

In [ ]:
!pip install -q langchain langchain-ollama

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = r.json().get("models", [])
    print(f"Ollama running — {len(models)} model(s) available")
    for m in models[:5]: print(f"  - {m['name']}")
except Exception as e:
    print(f"Cannot reach Ollama at {OLLAMA_BASE}: {e}")
    print("Start Ollama first: ollama serve")

## Step 2 — Sample Research Paper

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

paper = {
    "title": "Attention Is All You Need",
    "authors": "Vaswani et al., 2017",
    "abstract": """The dominant sequence transduction models are based on complex
recurrent or convolutional neural networks. We propose a new simple network
architecture, the Transformer, based solely on attention mechanisms,
dispensing with recurrence and convolutions entirely. The Transformer
generalizes well, achieving 28.4 BLEU on the WMT 2014 English-to-German
translation task.""",
    "sections": {
        "Introduction": "Recurrent models process sequences step by step, limiting "
            "parallelization. Attention mechanisms allow modeling dependencies "
            "regardless of distance. We propose the Transformer.",
        "Architecture": "The Transformer uses stacked self-attention and point-wise "
            "fully connected layers for both encoder and decoder. Multi-head "
            "attention allows the model to attend to different representation "
            "subspaces at different positions.",
        "Results": "The Transformer achieves 28.4 BLEU on English-to-German and "
            "41.8 BLEU on English-to-French, surpassing all previous models. "
            "Training took 3.5 days on 8 P100 GPUs.",
    }
}
print(f"Paper: {paper['title']}")
print(f"Sections: {list(paper['sections'].keys())}")

## Step 3 — Section-by-Section Explanation

In [ ]:
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research paper explainer. Explain this section "
     "in plain, beginner-friendly English. Use analogies where helpful."),
    ("human", "Paper: {title}\nSection: {section}\nContent: {content}\n\nExplain:")
])
explain_chain = explain_prompt | llm | StrOutputParser()

for section, content in paper["sections"].items():
    explanation = explain_chain.invoke({
        "title": paper["title"], "section": section, "content": content
    })
    print(f"\n### {section}")
    print(f"Original: {content[:80]}...")
    print(f"Explained: {explanation}")

## Step 4 — Multi-Level Explanations

In [ ]:
level_prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain this abstract at the {level} level. "
     "Beginner: use analogies, no jargon. "
     "Practitioner: assume ML basics. "
     "Expert: technical depth, cite specifics."),
    ("human", "Abstract: {abstract}")
])
level_chain = level_prompt | llm | StrOutputParser()

for level in ["beginner", "practitioner", "expert"]:
    result = level_chain.invoke({"abstract": paper["abstract"], "level": level})
    print(f"\n=== {level.upper()} ===")
    print(result)

## Step 5 — Glossary Extraction

In [ ]:
glossary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract technical terms from this paper content and define "
     "each in one simple sentence. Format: TERM: definition"),
    ("human", "Title: {title}\nAbstract: {abstract}\nContent: {content}")
])
glossary_chain = glossary_prompt | llm | StrOutputParser()

all_content = paper["abstract"] + " " + " ".join(paper["sections"].values())
glossary = glossary_chain.invoke({
    "title": paper["title"], "abstract": paper["abstract"], "content": all_content
})
print("GLOSSARY:")
print(glossary)

## Step 6 — Key Takeaways

In [ ]:
takeaway_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate exactly 5 key takeaways from this paper. "
     "Each should be one sentence, actionable or insightful."),
    ("human", "Title: {title}\nAbstract: {abstract}\nResults: {results}")
])
takeaway_chain = takeaway_prompt | llm | StrOutputParser()

takeaways = takeaway_chain.invoke({
    "title": paper["title"],
    "abstract": paper["abstract"],
    "results": paper["sections"]["Results"]
})
print("KEY TAKEAWAYS:")
print(takeaways)

## Limitations

1. **No PDF parsing** — uses pre-extracted text for simplicity.
2. **Section boundaries** must be manually provided.
3. **Long papers** need chunking before summarization.
4. **Quality** depends on the local model's knowledge.

## What You Learned

| Concept | What We Did |
|---|---|
| **Section-aware explanation** | Explain each part separately |
| **Difficulty levels** | Beginner / practitioner / expert |
| **Glossary extraction** | Auto-define technical terms |
| **Key takeaways** | Actionable summary cards |